# Projet complet — Attrition synthétique

**Question :** peut-on classer une attrition simulée à partir d'informations disponibles au moment de la prédiction ? **Pré-requis :** split stratifié, pipeline, probabilités et matrice de confusion. **Durée :** 2–3 h.

Une ligne représente un abonnement artificiel. `tenure_months`, `monthly_charge`, `support_calls_90d` et `contract` sont les features ; `churn` est le label. Le générateur injecte une relation statistique et du bruit : le modèle n'est pas adapté à de vrais clients ni à une décision individuelle.

**Protocole :** audit → baseline → pipeline logistique → Average Precision/ROC-AUC → seuil et erreurs → limites.

## 1. Audit de la cible et des variables

Vérifiez d'abord les taux de classes et les valeurs manquantes. Quand la classe positive est rare, l'accuracy d'un modèle qui prédit toujours la majorité peut sembler élevée. Les moyennes par label sont descriptives : elles ne prouvent pas qu'un tarif ou un contrat cause l'attrition.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from learning_hub.preprocessing import make_churn_data

features, target = make_churn_data(n_samples=1_000, random_state=31)
data = features.assign(churn=target)
print(data.head()); print(data.dtypes); print(data.isna().sum())
print('proportion positive:', round(data.churn.mean(), 3))
print(data.groupby('churn')[['tenure_months', 'monthly_charge', 'support_calls_90d']].mean().round(1))

### Visualisation explicative — voir le déséquilibre

Le graphique compare les effectifs par label. Il explique pourquoi une baseline de majorité peut obtenir une accuracy élevée tout en manquant tous les abonnements de la classe minoritaire.

In [ ]:
counts = data['churn'].value_counts().reindex([0, 1], fill_value=0)
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(['0 : reste', '1 : part'], counts.values, color=['tab:gray', 'tab:orange'])
for index, count in enumerate(counts.values):
    ax.text(index, count, str(count), ha='center', va='bottom')
ax.set(ylabel='Nombre de lignes', title='Classes du générateur synthétique', ylim=(0, counts.max() * 1.16))
plt.show()

## 2. Baseline et modèle probabiliste

La régression logistique calcule un logit `z = w·x+b`, puis `p = 1/(1+exp(-z))`. Cette `p` est un score entre 0 et 1 ; son interprétation comme probabilité fiable demande une vérification de calibration. Le modèle est régularisé et utilise `class_weight='balanced'`, ce qui change le compromis d'erreurs ; le seuil 0,5 ne devient pas automatiquement optimal.

La pipeline apprend médianes, échelles et catégories uniquement depuis `x_train`. Le split stratifié garde des proportions de labels proches ; il ne résout pas des dépendances par client ou temporelles. `Average Precision` résume precision–recall sur plusieurs seuils, souvent plus parlante quand la classe positive est minoritaire. ROC-AUC mesure le classement, pas le coût concret de chaque erreur.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from learning_hub.preprocessing import make_tabular_preprocessor

x_train, x_test, y_train, y_test = train_test_split(features, target, stratify=target, test_size=0.25, random_state=31)
baseline = DummyClassifier(strategy='most_frequent').fit(x_train, y_train)
pipeline = Pipeline([('prep', make_tabular_preprocessor(['tenure_months','monthly_charge','support_calls_90d'], ['contract'])), ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))]).fit(x_train, y_train)
probability = pipeline.predict_proba(x_test)[:, 1]
prediction = (probability >= 0.5).astype(int)
print('baseline accuracy:', round(baseline.score(x_test, y_test), 3))
print('Average Precision:', round(average_precision_score(y_test, probability), 3))
print('ROC-AUC:', round(roc_auc_score(y_test, probability), 3))
print(classification_report(y_test, prediction, zero_division=0))

## 3. Lire seuils et erreurs

La matrice de confusion compte les faux positifs et faux négatifs à `threshold=0.5`. La courbe precision–recall montre le compromis sur d'autres seuils. Le tableau d'erreurs permet de voir les caractéristiques des exemples difficiles. Choisir un seuil à partir du test rendrait le score final optimiste ; ce notebook le décrit seulement.

In [ ]:
ConfusionMatrixDisplay(confusion_matrix(y_test, prediction), display_labels=['retained','churn']).plot(cmap='Blues', values_format='d'); plt.show()
precision, recall, thresholds = precision_recall_curve(y_test, probability)
plt.plot(recall, precision); plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Test PR curve (descriptive only)'); plt.show()
errors = x_test.copy(); errors['actual'] = y_test; errors['predicted'] = prediction; errors['probability'] = probability
errors.loc[errors.actual != errors.predicted].head(10)

### Visualisation explicative — scores qui se chevauchent

Chaque histogramme montre les scores attribués aux vraies classes. Le chevauchement explique pourquoi aucun seuil ne sépare parfaitement les groupes. La ligne `0.5` est la décision affichée plus haut ; changer cette ligne échange généralement des faux positifs contre des faux négatifs.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for actual, label, color in [(0, 'Vrai 0', 'tab:gray'), (1, 'Vrai 1', 'tab:orange')]:
    ax.hist(probability[y_test == actual], bins=np.linspace(0, 1, 16), alpha=0.5, label=label, color=color)
ax.axvline(0.5, color='black', linestyle='--', label='Seuil 0.5')
ax.set(xlabel='Score du modèle pour la classe 1', ylabel='Nombre de lignes', title='Distribution des scores par vraie classe')
ax.legend(); plt.show()

## 4. Conclusion et suite

Rapportez la prévalence de la classe positive, l'Average Precision, la matrice de confusion et un type d'erreur à examiner. La baseline d'accuracy et la pipeline ne répondent pas exactement à la même question : donnez aussi une métrique adaptée à la classe rare. Le résultat décrit uniquement ce générateur et ce split.

Avant tout usage réel : vérifier consentement et qualité des données, fuite temporelle, représentativité, calibration des scores, coût des interventions et suivi de performance après déploiement. Une amélioration à tester ici serait de choisir le seuil en validation selon un coût explicite.